In [5]:
import os
# os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import math
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import scipy as sp
import tqdm

import warnings
warnings.filterwarnings("ignore", message=".*KMeans is known to have a memory leak.*")

from scipy.sparse import csr_matrix, diags, eye
from scipy.sparse.linalg import eigsh, lobpcg
from collections import defaultdict
from sklearn.metrics import normalized_mutual_info_score

import time

from utils import *

In [6]:
num_block = 10
block_size = 100000
p = 0.002
q = 0.00155

In [ ]:

random_state = int((time.time_ns() / 100000) % 100)
G, H = SBM(block_size=block_size, k=num_block, p=p, q=q)

y = [G.nodes[i]['block'] for i in G.nodes()]
print("random state = ",random_state)
# W_G = build_knn_graph(np.reshape(X, (len(X), -1)), k=3, use_gaussian=False, sigma=2)
W_G = nx.to_scipy_sparse_array(G)
W_H = nx.to_scipy_sparse_array(H)
# G = nx.from_scipy_sparse_matrix(W_G)

deg_g = np.array(W_G.sum(axis=1)).ravel()
normalized_Lap_G = normalized_laplacian_from_D_sp(W_G, (deg_g))
sc_labels, _, _ = spectral_clustering(
    normalized_Lap_G, 
    n_clusters=num_block,
    random_state=random_state
)
sc_ari = adjusted_rand_score(y, sc_labels)
sc_nmi = normalized_mutual_info_score(y, sc_labels)
sc_cut = max(conductance_all_clusters(G, sc_labels).values())
print("Spectral Clustering Results:")
print("ARI between clustering and true labels:", sc_ari)
print("NMI between clustering and true labels:", sc_nmi)
print("Spectral clustering cut: ", sc_cut)

aris = []
nmis = []
aris.append(sc_ari)
nmis.append(sc_nmi)
cut_gh = []
cut_g = []
cut_g.append(sc_cut)
cut_gh.append(None)
pred_labels = sc_labels

    # W_H = build_cannot_link_graph(pred_labels, n_cl=min(n_neighbors + i, neighbor_max), random_state=random_state)
    # H = nx.from_scipy_sparse_matrix(W_H)
pred_labels, vals, vecs = iterative_clustering(W_G, W_H, k = num_block, exact=False, eps = 0.9 ,random_seed=random_state)
# print("vals   ", vals)
generalized_spectral_clustering_ari = adjusted_rand_score(y, pred_labels)
generalized_spectral_clustering_nmi = normalized_mutual_info_score(y, pred_labels)
print(f"Constrained Spectral Clustering ARI: {generalized_spectral_clustering_ari:.4f}")
print(f"Constrained Spectral Clustering NMI: {generalized_spectral_clustering_nmi:.4f}")
    


In [ ]:

# ── Wang et al. (2014) Algorithm 2: CSC for K-way Partition ─────────────────
# "On Constrained Spectral Clustering and Its Applications"
# Wang, Qian, Davidson — Data Mining and Knowledge Discovery, 2014
#
# Inputs:  A (affinity), Q (constraint: +1 must-link, -1 cannot-link), β, K
# Core:    solve generalized eigenproblem  L̄ v = λ (Q̄ − β/vol · I) v
# Output:  K-means on D^{−1/2} V*  (K−1 smallest-positive-λ eigenvectors)
#
# Implementation note: the RHS matrix M = Q̄ − β/vol·I is generally indefinite.
# We use a Rayleigh-Ritz projection onto the positive eigenspace of Q̄ to convert
# it into a small dense positive-definite generalized eigenproblem.

import scipy.linalg as sla

def wang_algorithm2(A, Q, beta, K, n_extra=20, random_state=0, eps=1e-10):
    """
    Wang et al. (2014) Algorithm 2 — Constrained Spectral Clustering (K-way).

    Parameters
    ----------
    A       : sparse (N×N)  affinity matrix
    Q       : sparse (N×N)  constraint matrix  (+1 must-link, −1 cannot-link)
    beta    : float          threshold parameter  (must satisfy β < λ_{K-1}·vol)
    K       : int            number of clusters
    n_extra : int            extra Ritz vectors beyond K−1 (improves accuracy)
    """
    N = A.shape[0]

    # ── Step 1: volume and degree ────────────────────────────────────────────
    vol        = float(A.sum())
    deg        = np.array(A.sum(axis=1)).ravel()
    inv_sqrt   = 1.0 / np.sqrt(np.maximum(deg, eps))
    D_inv_sqrt = diags(inv_sqrt)

    # ── Step 2: normalized Laplacian L̄ and normalized constraint matrix Q̄ ──
    I_sp  = eye(N, format='csr')
    L_bar = I_sp - D_inv_sqrt @ A @ D_inv_sqrt     # L̄ = I − D^{−½} A D^{−½}
    Q_bar = D_inv_sqrt @ Q @ D_inv_sqrt            # Q̄ = D^{−½} Q D^{−½}

    # ── Step 3: top eigenvalues of Q̄ (need (K−1)-th largest for Step 4) ────
    n_q    = min(K - 1 + n_extra, N - 2)
    q_vals, q_vecs = eigsh(Q_bar, k=n_q, which='LM')
    order  = np.argsort(q_vals)[::-1]              # sort descending
    q_vals = q_vals[order];  q_vecs = q_vecs[:, order]

    lambda_K1 = q_vals[K - 2]                      # (K−1)-th largest eigenvalue

    # ── Step 4: feasibility check ────────────────────────────────────────────
    if beta >= lambda_K1 * vol:
        raise ValueError(
            f"Infeasible: β={beta:.4f} ≥ λ_{{K-1}}·vol={lambda_K1*vol:.4f}. "
            f"Choose β < {lambda_K1*vol:.6f}."
        )

    # ── Steps 5-7 via Rayleigh-Ritz in Q̄'s positive eigenspace ─────────────
    # M = Q̄ − (β/vol)·I  is positive definite where q_vals > β/vol.
    # We project L̄ and M onto that subspace to get a small dense problem.
    threshold = beta / vol
    pos_mask  = q_vals > threshold
    V         = q_vecs[:, pos_mask]               # basis  (N × m)
    m_diag    = q_vals[pos_mask] - threshold      # M's diagonal in subspace (all > 0)

    if V.shape[1] < K - 1:
        raise ValueError(
            f"Only {V.shape[1]} feasible Ritz vectors; need K−1={K-1}. "
            f"Reduce β or increase n_extra."
        )

    # Dense projected problem: L_sub v = λ M_sub v  (m × m)
    L_sub = V.T @ (L_bar @ V)                     # Vᵀ L̄ V
    M_sub = np.diag(m_diag)                        # Vᵀ M V (diagonal)
    sub_vals, sub_vecs = sla.eigh(L_sub, M_sub)    # all real eigenvalues

    # ── Step 6: remove non-positive eigenvalues; normalize v ← v/‖v‖ · √vol ─
    pos_idx = np.where(sub_vals > 1e-8)[0]
    if len(pos_idx) < K - 1:                       # fallback: take K−1 largest
        pos_idx = np.argsort(sub_vals)[-(K - 1):]

    # ── Step 7: pick K−1 eigenvectors with smallest positive λ ──────────────
    #   (minimises trace(Vᵀ L̄ V) — the K−1 lowest-cut directions)
    best_k1    = np.argsort(sub_vals[pos_idx])[:K - 1]
    sel_idx    = pos_idx[best_k1]
    V_star     = V @ sub_vecs[:, sel_idx]          # back to N-dim  (N × K−1)

    # Normalise so that vᵀv = vol
    norms  = np.linalg.norm(V_star, axis=0, keepdims=True)
    V_star = V_star / np.maximum(norms, eps) * np.sqrt(vol)

    # ── Step 8: K-means on D^{−½} V* ────────────────────────────────────────
    embedding = D_inv_sqrt @ V_star
    km     = KMeans(n_clusters=K, n_init=10, random_state=random_state)
    labels = km.fit_predict(embedding)

    return labels, sub_vals[sel_idx], V_star


# ── Construct constraint matrix Q ────────────────────────────────────────────
# SBM: W_G edges are within-cluster-biased  → must-link   (+1)
#      W_H edges are between-cluster-biased → cannot-link  (−1)
#  Q_ij = +1 if (i,j) ∈ ML,  −1 if (i,j) ∈ CL,  0 otherwise
Q_wang = (W_G - W_H).astype(float)

# ── Choose β ─────────────────────────────────────────────────────────────────
# β = 0  →  M = Q̄  (maximum constraint enforcement; always feasible if Q̄ has
#           K−1 positive eigenvalues, which holds when the constraints are
#           informative about cluster structure)
beta_wang = 0.0

# ── Run Wang et al. Algorithm 2 ──────────────────────────────────────────────
wang_labels, wang_evals, wang_Vstar = wang_algorithm2(
    A=W_G,
    Q=Q_wang,
    beta=beta_wang,
    K=num_block,
    n_extra=20,
    random_state=random_state
)

wang_ari = adjusted_rand_score(y, wang_labels)
wang_nmi = normalized_mutual_info_score(y, wang_labels)

print("Wang et al. (2014) Algorithm 2 — Constrained Spectral Clustering")
print(f"  β     = {beta_wang}")
print(f"  ARI   = {wang_ari:.4f}")
print(f"  NMI   = {wang_nmi:.4f}")

# ── Comparison table ──────────────────────────────────────────────────────────
print()
print(f"{'Method':<45} {'ARI':>7} {'NMI':>7}")
print("-" * 60)
print(f"{'Spectral Clustering (unconstrained)':<45} {sc_ari:>7.4f} {sc_nmi:>7.4f}")
print(f"{'Wang et al. (2014) Algorithm 2':<45} {wang_ari:>7.4f} {wang_nmi:>7.4f}")
print(f"{'Our Constrained SC':<45} {generalized_spectral_clustering_ari:>7.4f} {generalized_spectral_clustering_nmi:>7.4f}")


Wang et al. (2014) Algorithm 2 — Constrained Spectral Clustering
  β     = 0.0
  ARI   = 0.0001
  NMI   = 0.0004

Method                                            ARI     NMI
------------------------------------------------------------
Spectral Clustering (unconstrained)            0.0000  0.0002
Wang et al. (2014) Algorithm 2                 0.0001  0.0004
Our Constrained SC                             0.0001  0.0003
